# 19 — G1: small ST-GCN versus V6-C TCN

This is the controlled model-family comparison from the fixed experiment
backbone. It preserves the V6-A.1 group split, Notebook 13 hard-positive
weights, V6-B train-only augmentation, sampler, seed, optimizer, early stopping,
and internal evaluation. It changes the recognition core from a temporal
convolutional network to a small spatial-temporal graph convolutional network.

The ST-GCN consumes the same normalized 17-joint pose windows. Each graph node
receives x/y coordinates and x/y velocity; the fixed adjacency matrix connects
anatomically related joints. This is a model-family comparison rather than a
single-feature ablation: graph message passing replaces V6-C's flattened bone
vectors and joint-angle channels.

The notebook automatically places the frozen V6-C metrics beside the G1 result.
Promotion requires a meaningful Macro F1 gain without harming unknown rejection,
not merely higher raw accuracy. No raw video or MediaPipe step runs here.

Use a **T4 runtime**. Output:
`calorie_g1_stgcn_comparison_report.zip`.


In [ ]:
%pip install -q scikit-learn seaborn pandas==2.2.3


In [ ]:
from pathlib import Path
import json
import random
import shutil
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from google.colab import files
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

SEED = 42
WINDOW = 64
STRIDE = 32
BATCH_SIZE = 48
MAX_EPOCHS = 40
PATIENCE = 8
GROUP_TEMPERATURE = 0.5  # frozen from V6-A.1
AUGMENTATION_CONFIG = {
    "horizontal_flip_probability": 0.5,
    "rotation_probability": 0.7,
    "max_rotation_degrees": 7.0,
    "speed_probability": 0.5,
    "speed_range": [0.85, 1.15],
    "frame_drop_probability": 0.35,
    "frame_drop_fraction_range": [0.05, 0.12],
    "coordinate_noise_probability": 0.8,
    "coordinate_noise_std": 0.008,
}
RETIRED_CLASSES = {"plank"}
UNREVIEWED_GROUPS = {
    "youtube:oLsonHQJOUc",
    "youtube:c4TomZjzobY",
    "youtube:W2Bm9dfQjbk",
}


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Experiment: G1 small ST-GCN comparison")
assert DEVICE.type == "cuda", "Select a T4 runtime before training."


## Load frozen landmark evidence and Notebook 13 weights

No video is downloaded and MediaPipe does not run here. The hard-positive CSV
contains `window_key` values for difficult supported windows mined from the v5
**training split only**. It supplies weights, not copied training examples.


In [ ]:
REPOSITORY_RAW = (
    "https://raw.githubusercontent.com/bariscelikk1/cALorie/"
    "feat/tcn-rebuild/ml/artifacts"
)
LANDMARK_ARCHIVES = [
    ("v1", "calorie_workoutfitness_landmarks_v1.zip"),
    ("v1", "calorie_mmfit_landmarks_v1.zip"),
    ("v1", "calorie_mmfit_unknown_v1.zip"),
    ("v2", "calorie_youtube_mixed_landmarks_v3.zip"),
    ("v3", "calorie_youtube_additional_landmarks_v1.zip"),
    ("v3", "calorie_youtube_training_landmarks_v4.zip"),
    ("v5", "calorie_workoutfitness_gate_hardneg_v1.zip"),
]


def download(url, destination):
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=300) as response, destination.open("wb") as output:
        shutil.copyfileobj(response, output)
    print(f"{destination.name}: {destination.stat().st_size / 1e6:.1f} MB")


zip_paths = []
for version, name in LANDMARK_ARCHIVES:
    destination = Path("/content") / name
    download(f"{REPOSITORY_RAW}/{version}/landmarks/{name}", destination)
    zip_paths.append(destination)

HARD_POSITIVE_PATH = Path("/content/hard_positive_windows.csv")
download(
    f"{REPOSITORY_RAW}/v5/hard_positive_mining/hard_positive_windows.csv",
    HARD_POSITIVE_PATH,
)
hard_positive_frame = pd.read_csv(HARD_POSITIVE_PATH)
assert hard_positive_frame.window_key.is_unique, "Hard-positive window keys must be unique."

V6A1_SPLIT_PATH = Path("/content/v6a1_split_assignments.csv")
download(
    f"{REPOSITORY_RAW}/v6a1/model/split_assignments.csv",
    V6A1_SPLIT_PATH,
)

V6C_METRICS_PATH = Path("/content/v6c_metrics.json")
download(
    f"{REPOSITORY_RAW}/v6c/model/metrics.json",
    V6C_METRICS_PATH,
)
V6C_METRICS = json.loads(V6C_METRICS_PATH.read_text())

SOURCE_ROOT = Path("/content/g1_stgcn_landmark_sources")
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir()
for index, archive_path in enumerate(zip_paths):
    destination = SOURCE_ROOT / f"source_{index}"
    destination.mkdir()
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(destination)

frames = []
for manifest_path in SOURCE_ROOT.rglob("manifest.csv"):
    frame = pd.read_csv(manifest_path).fillna("")
    frame["npz_path"] = frame["output_path"].map(
        lambda value: str(manifest_path.parent / value)
    )
    frames.append(frame)
assert frames, "No manifest.csv was found."
manifest = pd.concat(frames, ignore_index=True)
manifest = manifest[
    (manifest.status == "ok")
    & manifest.npz_path.map(lambda value: Path(value).exists())
].copy()
if "group_id" not in manifest or manifest.group_id.eq("").any():
    manifest["group_id"] = manifest.apply(
        lambda row: f"{row['source_dataset']}:{row['clip_id']}", axis=1
    )
print("Valid clips:", len(manifest), "Groups:", manifest.group_id.nunique())
display(manifest.canonical_class.value_counts().sort_index().rename("clips"))


## Rebuild windows and load the frozen V6-A.1 split

The exported V6-A.1 `split_assignments.csv` is the authority. Every source
group receives exactly the same train, validation, or test assignment used by
V6-B and V6-C. The active taxonomy still maps archived plank to `unknown`.


In [ ]:
def resample_sequence(sequence, source_fps, target_fps=15):
    if not source_fps or abs(float(source_fps) - target_fps) < 0.1:
        return sequence
    duration = (len(sequence) - 1) / float(source_fps)
    target_length = max(2, int(round(duration * target_fps)) + 1)
    old_x = np.linspace(0, duration, len(sequence))
    new_x = np.linspace(0, duration, target_length)
    flat = sequence.reshape(len(sequence), -1)
    sampled = np.stack([
        np.interp(new_x, old_x, flat[:, index])
        for index in range(flat.shape[1])
    ], axis=1)
    return sampled.reshape(target_length, 17, 4).astype(np.float32)


windows, original_labels, groups = [], [], []
clip_ids, sources, starts = [], [], []
for row in manifest.itertuples(index=False):
    with np.load(row.npz_path, allow_pickle=False) as item:
        sequence = item["landmarks"].astype(np.float32)
    if sequence.ndim != 3 or sequence.shape[1:] != (17, 4):
        print("Skipping unexpected shape:", row.npz_path, sequence.shape)
        continue
    sequence = resample_sequence(sequence, getattr(row, "source_fps", 15) or 15)
    clip_starts = list(range(0, max(1, len(sequence) - WINDOW + 1), STRIDE))
    if len(sequence) > WINDOW and clip_starts[-1] != len(sequence) - WINDOW:
        clip_starts.append(len(sequence) - WINDOW)
    for start in clip_starts:
        window = sequence[start:start + WINDOW]
        if len(window) < WINDOW:
            window = np.concatenate([
                window, np.repeat(window[-1:], WINDOW - len(window), axis=0)
            ])
        windows.append(window)
        original_labels.append(row.canonical_class)
        groups.append(row.group_id)
        clip_ids.append(row.clip_id)
        sources.append(row.source_dataset)
        starts.append(start)

X = np.asarray(windows, dtype=np.float32)
original_labels = np.asarray(original_labels)
groups = np.asarray(groups)
clip_ids = np.asarray(clip_ids)
sources = np.asarray(sources)
starts = np.asarray(starts, dtype=np.int64)
window_keys = np.asarray([
    f"{source}|{group}|{clip}|{start}"
    for source, group, clip, start in zip(sources, groups, clip_ids, starts)
])

# Reuse the exact group assignment exported by V6-A.1.
frozen_split = pd.read_csv(V6A1_SPLIT_PATH)
assert frozen_split.groupby("group_id")["split"].nunique().max() == 1
group_to_split = (
    frozen_split[["group_id", "split"]]
    .drop_duplicates("group_id")
    .set_index("group_id")["split"]
    .to_dict()
)
assigned_split = np.asarray([group_to_split.get(group, "") for group in groups])
known_split_mask = assigned_split != ""
missing_groups = set(groups[~known_split_mask])
assert missing_groups <= UNREVIEWED_GROUPS, (
    "Unexpected groups absent from the frozen split: " + str(sorted(missing_groups))
)
train_idx = np.flatnonzero(assigned_split == "train")
val_idx = np.flatnonzero(assigned_split == "validation")
test_idx = np.flatnonzero(assigned_split == "test")

labels = np.where(np.isin(original_labels, list(RETIRED_CLASSES)), "unknown", original_labels)
ALL_CLASSES = sorted(np.unique(labels).tolist())
SUPPORTED_CLASSES = [name for name in ALL_CLASSES if name != "unknown"]
assert len(SUPPORTED_CLASSES) == 14, SUPPORTED_CLASSES
all_to_id = {name: index for index, name in enumerate(ALL_CLASSES)}
supported_to_id = {name: index for index, name in enumerate(SUPPORTED_CLASSES)}
y_all = np.asarray([all_to_id[name] for name in labels], dtype=np.int64)
y_gate = (labels != "unknown").astype(np.int64)
y_exercise = np.asarray([supported_to_id.get(name, -1) for name in labels], dtype=np.int64)

assert not (set(groups[train_idx]) & set(groups[val_idx]))
assert not (set(groups[train_idx]) & set(groups[test_idx]))
assert not (set(groups[val_idx]) & set(groups[test_idx]))
assert not (set(groups[train_idx]) & UNREVIEWED_GROUPS)
assert set(window_keys[known_split_mask]) == set(frozen_split.window_key), (
    "Reviewed window evidence changed since V6-A.1."
)
print("Excluded unresolved groups:", sorted(missing_groups))

print("Windows:", X.shape)
print("Frozen groups:", len(group_to_split))
print("Active supported classes:", SUPPORTED_CLASSES)
print("Retired plank windows mapped to unknown:", int((original_labels == "plank").sum()))
for name, indices in [("train", train_idx), ("validation", val_idx), ("test", test_idx)]:
    print(name, "windows=", len(indices), "groups=", len(np.unique(groups[indices])))
    display(pd.Series(labels[indices]).value_counts().sort_index().rename("windows"))


## Tempered hierarchical sampling and hard-positive weighting

Sampling first assigns equal probability to each class. Inside a class, each
source group receives probability proportional to `window_count ** 0.5`. This is
the midpoint between failed V6-A equal-group sampling (`** 0`) and raw window
sampling (`** 1`): long videos are restrained without making a one-window group
as influential as a large group. Notebook 13 hard-positive modifiers are
normalized inside the same class/group, so they change which windows are seen
without changing that group's total probability.

For the gate, “class” means supported or unknown. For the exercise model it
means one of the 14 active movements. `WeightedRandomSampler` then draws the
requested class -> group -> window distribution.


In [ ]:
hard_weight_by_key = dict(zip(
    hard_positive_frame.window_key,
    hard_positive_frame.recommended_sample_weight,
))
hard_positive_weights = np.asarray([
    float(hard_weight_by_key.get(key, 1.0)) for key in window_keys
], dtype=np.float64)
# Retired plank is now unknown and cannot receive a supported hard-positive boost.
hard_positive_weights[labels == "unknown"] = 1.0
matched_training_keys = (
    set(window_keys[train_idx][labels[train_idx] != "unknown"])
    & set(hard_weight_by_key)
)
assert not (set(window_keys[val_idx]) & set(hard_weight_by_key)), (
    "Notebook 13 weights unexpectedly intersect validation."
)
assert not (set(window_keys[test_idx]) & set(hard_weight_by_key)), (
    "Notebook 13 weights unexpectedly intersect test."
)
print("Notebook 13 keys:", len(hard_weight_by_key))
print("Matched active training keys:", len(matched_training_keys))
print("Upweighted active training windows:", int((hard_positive_weights[train_idx] > 1).sum()))


def hierarchical_window_weights(indices, hierarchy_labels, extra_weights=None):
    table = pd.DataFrame({
        "index": indices,
        "hierarchy_label": hierarchy_labels[indices],
        "group": groups[indices],
    })
    class_count = table["hierarchy_label"].nunique()
    group_sizes = table.groupby(
        ["hierarchy_label", "group"]
    )["index"].transform("count").astype(float)
    table["group_mass"] = group_sizes.pow(GROUP_TEMPERATURE)
    group_table = table[["hierarchy_label", "group"]].copy()
    group_table["group_mass"] = table["group_mass"]
    class_group_mass = (
        group_table.drop_duplicates(["hierarchy_label", "group"])
        .groupby("hierarchy_label")["group_mass"].sum()
    )
    table["extra_weight"] = (
        1.0 if extra_weights is None else extra_weights[table["index"].to_numpy()]
    )
    table["pair_extra_total"] = table.groupby(
        ["hierarchy_label", "group"]
    )["extra_weight"].transform("sum")
    weights = []
    for row in table.itertuples(index=False):
        weight = (
            1.0 / class_count
            * row.group_mass / class_group_mass.loc[row.hierarchy_label]
            * row.extra_weight / row.pair_extra_total
        )
        weights.append(weight)
    weights = np.asarray(weights, dtype=np.float64)
    return weights / weights.mean()


# Use strings here so the audit table remains human-readable.
gate_hierarchy = np.where(y_gate == 1, "supported", "unknown")
gate_train_weights = hierarchical_window_weights(
    train_idx, gate_hierarchy, hard_positive_weights
)
exercise_train_idx = train_idx[y_exercise[train_idx] >= 0]
exercise_val_idx = val_idx[y_exercise[val_idx] >= 0]
exercise_test_idx = test_idx[y_exercise[test_idx] >= 0]
exercise_train_weights = hierarchical_window_weights(
    exercise_train_idx, labels
)

sampler_audit = pd.DataFrame({
    "window_key": window_keys[train_idx],
    "canonical_class": labels[train_idx],
    "original_class": original_labels[train_idx],
    "group_id": groups[train_idx],
    "gate_target": gate_hierarchy[train_idx],
    "hard_positive_weight": hard_positive_weights[train_idx],
    "gate_sampler_weight": gate_train_weights,
})
exercise_weight_map = dict(zip(
    window_keys[exercise_train_idx], exercise_train_weights
))
sampler_audit["exercise_sampler_weight"] = sampler_audit.window_key.map(
    exercise_weight_map
).fillna(0.0)

display(sampler_audit.groupby("gate_target").agg(
    windows=("window_key", "count"),
    groups=("group_id", "nunique"),
    total_sampler_weight=("gate_sampler_weight", "sum"),
).round(3))
display(sampler_audit[sampler_audit.canonical_class != "unknown"].groupby(
    "canonical_class"
).agg(
    windows=("window_key", "count"),
    groups=("group_id", "nunique"),
    total_exercise_weight=("exercise_sampler_weight", "sum"),
).round(3))


In [ ]:
LEFT_RIGHT_PAIRS = [
    (1, 2), (3, 4), (5, 6), (7, 8),
    (9, 10), (11, 12), (13, 14), (15, 16),
]


def augment_pose_window(sample, force_all=False):
    sample = sample.copy()
    config = AUGMENTATION_CONFIG

    if force_all or np.random.random() < config["horizontal_flip_probability"]:
        sample[:, :, 0] *= -1.0
        for left, right in LEFT_RIGHT_PAIRS:
            sample[:, [left, right], :] = sample[:, [right, left], :]

    if force_all or np.random.random() < config["rotation_probability"]:
        degrees = np.random.uniform(
            -config["max_rotation_degrees"], config["max_rotation_degrees"]
        )
        radians = np.deg2rad(degrees)
        cosine, sine = np.cos(radians), np.sin(radians)
        x = sample[:, :, 0].copy()
        y = sample[:, :, 1].copy()
        sample[:, :, 0] = cosine * x - sine * y
        sample[:, :, 1] = sine * x + cosine * y

    if force_all or np.random.random() < config["speed_probability"]:
        speed = np.random.uniform(*config["speed_range"])
        center = (WINDOW - 1) / 2.0
        positions = (np.arange(WINDOW) - center) * speed + center
        flat = sample.reshape(WINDOW, -1)
        flat = np.stack([
            np.interp(positions, np.arange(WINDOW), flat[:, column])
            for column in range(flat.shape[1])
        ], axis=1)
        sample = flat.reshape(WINDOW, 17, 4).astype(np.float32)

    if force_all or np.random.random() < config["frame_drop_probability"]:
        fraction = np.random.uniform(*config["frame_drop_fraction_range"])
        count = max(1, int(round(WINDOW * fraction)))
        dropped = np.random.choice(np.arange(1, WINDOW), size=count, replace=False)
        sample[dropped] = sample[dropped - 1]

    if force_all or np.random.random() < config["coordinate_noise_probability"]:
        sample[:, :, :2] += np.random.normal(
            0, config["coordinate_noise_std"], sample[:, :, :2].shape
        ).astype(np.float32)
    return sample.astype(np.float32)



NODE_FEATURE_CONFIG = {
    "coordinates_xy_per_joint": 2,
    "velocity_xy_per_joint": 2,
}
NODE_FEATURES = sum(NODE_FEATURE_CONFIG.values())


def pose_to_graph_features(sample):
    """Convert [time, 17, 4] pose to [channels, time, joints]."""
    xy = sample[:, :, :2].astype(np.float32)
    velocity = np.zeros_like(xy)
    velocity[1:] = xy[1:] - xy[:-1]
    features = np.concatenate([xy, velocity], axis=2).transpose(2, 0, 1)
    assert features.shape == (NODE_FEATURES, WINDOW, 17), features.shape
    assert np.isfinite(features).all()
    return features.astype(np.float32)


class PoseWindowDataset(Dataset):
    def __init__(self, indices, targets, training=False):
        self.indices = np.asarray(indices)
        self.targets = targets
        self.training = training

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        actual = self.indices[item]
        sample = X[actual].copy()
        if self.training:
            sample = augment_pose_window(sample)
        features = pose_to_graph_features(sample)
        return torch.from_numpy(features), torch.tensor(
            self.targets[actual], dtype=torch.long
        )


def make_loaders(targets, train_indices, val_indices, test_indices, train_weights, seed):
    generator = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(
        torch.as_tensor(train_weights, dtype=torch.double),
        len(train_weights), replacement=True, generator=generator,
    )
    return (
        DataLoader(
            PoseWindowDataset(train_indices, targets, True),
            batch_size=BATCH_SIZE, sampler=sampler,
        ),
        DataLoader(
            PoseWindowDataset(val_indices, targets),
            batch_size=BATCH_SIZE, shuffle=False,
        ),
        DataLoader(
            PoseWindowDataset(test_indices, targets),
            batch_size=BATCH_SIZE, shuffle=False,
        ),
    )


gate_loaders = make_loaders(
    y_gate, train_idx, val_idx, test_idx, gate_train_weights, SEED
)
exercise_loaders = make_loaders(
    y_exercise, exercise_train_idx, exercise_val_idx, exercise_test_idx,
    exercise_train_weights, SEED + 1,
)
print("Node feature blocks:", NODE_FEATURE_CONFIG)
print("Input channels per joint:", NODE_FEATURES)
print("Gate split sizes:", [len(loader.dataset) for loader in gate_loaders])
print("Exercise split sizes:", [len(loader.dataset) for loader in exercise_loaders])


## Audit the augmentation before training

The preview applies every transformation once to a real training window. Check
that limbs remain connected, left/right swapping is anatomical, and motion is
perturbed without becoming a different exercise. The image is included in the
report for reproducibility.


In [ ]:
SKELETON_EDGES = [
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12),
    (11, 13), (13, 15), (12, 14), (14, 16),
]
preview_candidates = train_idx[labels[train_idx] == "squat"]
preview_index = int(preview_candidates[0] if len(preview_candidates) else train_idx[0])
original_preview = X[preview_index].copy()
random_state = np.random.get_state()
np.random.seed(SEED + 1000)
augmented_preview = augment_pose_window(original_preview, force_all=True)
np.random.set_state(random_state)

fig, axes = plt.subplots(2, 6, figsize=(15, 6))
frame_indices = np.linspace(0, WINDOW - 1, 6, dtype=int)
for row, (title, sequence) in enumerate([
    ("original", original_preview), ("augmented", augmented_preview)
]):
    for axis, frame_index in zip(axes[row], frame_indices):
        xy = sequence[frame_index, :, :2]
        for left, right in SKELETON_EDGES:
            axis.plot(
                [xy[left, 0], xy[right, 0]],
                [-xy[left, 1], -xy[right, 1]],
                color="tab:blue", linewidth=1.4,
            )
        axis.scatter(xy[:, 0], -xy[:, 1], s=12, color="tab:red")
        axis.set_aspect("equal")
        axis.axis("off")
        axis.set_title(f"{title} t={frame_index}")
fig.suptitle(f"V6-B augmentation audit | {labels[preview_index]} | {window_keys[preview_index]}")
fig.tight_layout()
AUGMENTATION_PREVIEW_PATH = Path("/content/v6b_augmentation_preview.png")
fig.savefig(AUGMENTATION_PREVIEW_PATH, dpi=170, bbox_inches="tight")
plt.show()
print("Preview:", AUGMENTATION_PREVIEW_PATH)


## Small spatial-temporal graph convolutional network

The 17-node adjacency matrix contains self-connections and the same anatomical
edges used in pose previews. Each block first mixes information across connected
joints, then applies a temporal convolution. Global pooling produces one vector
for the gate or exercise classifier.

The graph is fixed; a small learned edge-importance matrix lets each block
strengthen or weaken existing anatomical connections. The two-stage gate and
14-class classifier remain separate, matching V6-C.


In [ ]:
SKELETON_EDGES = [
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12),
    (11, 13), (13, 15), (12, 14), (14, 16),
]


def normalized_adjacency(num_nodes=17):
    adjacency = np.eye(num_nodes, dtype=np.float32)
    for first, second in SKELETON_EDGES:
        adjacency[first, second] = 1.0
        adjacency[second, first] = 1.0
    degree = adjacency.sum(axis=1)
    inverse_sqrt = np.diag(1.0 / np.sqrt(np.maximum(degree, 1e-6)))
    return (inverse_sqrt @ adjacency @ inverse_sqrt).astype(np.float32)


class STGCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, temporal_kernel=9, dropout=0.2):
        super().__init__()
        self.register_buffer("adjacency", torch.from_numpy(normalized_adjacency()))
        self.edge_importance = nn.Parameter(torch.ones(17, 17))
        self.spatial = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.temporal = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(
                out_channels, out_channels,
                kernel_size=(temporal_kernel, 1),
                padding=((temporal_kernel - 1) // 2, 0),
            ),
            nn.BatchNorm2d(out_channels),
            nn.Dropout(dropout),
        )
        self.residual = (
            nn.Conv2d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels else nn.Identity()
        )

    def forward(self, x):
        weighted_adjacency = self.adjacency * self.edge_importance
        graph_mixed = torch.einsum("bctv,vw->bctw", x, weighted_adjacency)
        spatial = self.spatial(graph_mixed)
        return torch.relu(self.temporal(spatial) + self.residual(x))


class SmallSTGCN(nn.Module):
    def __init__(self, input_channels, num_classes):
        super().__init__()
        channels = [32, 64, 96, 128]
        blocks, current = [], input_channels
        for output in channels:
            blocks.append(STGCNBlock(current, output))
            current = output
        self.features = nn.Sequential(*blocks)
        self.classifier = nn.Linear(current, num_classes)

    def forward(self, x):
        features = self.features(x).mean(dim=(2, 3))
        return self.classifier(features)


A = normalized_adjacency()
print("Graph nodes:", A.shape[0], "directed non-zero entries:", int((A > 0).sum()))


In [ ]:
def predict_model(model, loader):
    model.eval()
    truths, probabilities = [], []
    with torch.no_grad():
        for features, targets in loader:
            logits = model(features.to(DEVICE))
            truths.extend(targets.numpy())
            probabilities.extend(logits.softmax(1).cpu().numpy())
    probabilities = np.asarray(probabilities)
    return np.asarray(truths), probabilities.argmax(1), probabilities


def train_model(name, num_classes, loaders, seed):
    seed_everything(seed)
    train_loader, val_loader, _ = loaders
    model = SmallSTGCN(NODE_FEATURES, num_classes).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=3, factor=0.5
    )
    best_path = Path(f"/content/best_g1_stgcn_{name}.pt")
    best_f1, stale, history = -1.0, 0, []

    def run_epoch(loader, training):
        model.train(training)
        losses, truths, predictions = [], [], []
        context = torch.enable_grad() if training else torch.no_grad()
        with context:
            for features, targets in loader:
                features, targets = features.to(DEVICE), targets.to(DEVICE)
                if training:
                    optimizer.zero_grad(set_to_none=True)
                logits = model(features)
                loss = criterion(logits, targets)
                if training:
                    loss.backward()
                    optimizer.step()
                losses.append(loss.item() * len(targets))
                truths.extend(targets.cpu().numpy())
                predictions.extend(logits.argmax(1).cpu().numpy())
        return (
            sum(losses) / len(truths),
            f1_score(truths, predictions, average="macro", zero_division=0),
        )

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, train_f1 = run_epoch(train_loader, True)
        val_loss, val_f1 = run_epoch(val_loader, False)
        scheduler.step(val_f1)
        history.append({
            "model": name, "epoch": epoch,
            "train_loss": train_loss, "train_macro_f1": train_f1,
            "val_loss": val_loss, "val_macro_f1": val_f1,
        })
        print(
            f"{name} {epoch:02d} train loss={train_loss:.4f} F1={train_f1:.3f} "
            f"| val loss={val_loss:.4f} F1={val_f1:.3f}"
        )
        if val_f1 > best_f1 + 1e-4:
            best_f1, stale = val_f1, 0
            torch.save(model.state_dict(), best_path)
        else:
            stale += 1
            if stale >= PATIENCE:
                print(name, "early stopping")
                break

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    return model, history, best_f1


gate_model, gate_history, gate_best_f1 = train_model(
    "gate", 2, gate_loaders, SEED
)
exercise_model, exercise_history, exercise_best_f1 = train_model(
    "exercise", len(SUPPORTED_CLASSES), exercise_loaders, SEED + 1
)
GATE_PARAMETER_COUNT = sum(parameter.numel() for parameter in gate_model.parameters())
EXERCISE_PARAMETER_COUNT = sum(parameter.numel() for parameter in exercise_model.parameters())
print("Best validation F1 — gate:", round(gate_best_f1, 4))
print("Best validation F1 — exercise:", round(exercise_best_f1, 4))
print("Parameters — gate:", f"{GATE_PARAMETER_COUNT:,}", "exercise:", f"{EXERCISE_PARAMETER_COUNT:,}")


## Interim validation-only threshold

This repeats V6-C's validation sweep for the new ST-GCN outputs. It is required
for a fair internal comparison but does not replace Notebook 17's independent,
provisional creator calibration.


In [ ]:
_, _, gate_val_probability = predict_model(gate_model, gate_loaders[1])


def full_probabilities(indices):
    features = np.stack([
        pose_to_graph_features(X[index]) for index in indices
    ])
    gate_parts, exercise_parts = [], []
    gate_model.eval()
    exercise_model.eval()
    with torch.no_grad():
        for start in range(0, len(features), 256):
            batch = torch.from_numpy(features[start:start + 256]).to(DEVICE)
            gate_parts.append(gate_model(batch).softmax(1).cpu().numpy())
            exercise_parts.append(exercise_model(batch).softmax(1).cpu().numpy())
    return np.concatenate(gate_parts), np.concatenate(exercise_parts)


_, exercise_val_probabilities = full_probabilities(val_idx)
exercise_val_predictions = exercise_val_probabilities.argmax(1)
threshold_rows = []
for threshold in np.linspace(0.05, 0.95, 37):
    end_predictions = np.asarray([
        SUPPORTED_CLASSES[class_id] if probability >= threshold else "unknown"
        for probability, class_id in zip(
            gate_val_probability[:, 1], exercise_val_predictions
        )
    ])
    supported_mask = labels[val_idx] != "unknown"
    unknown_mask = ~supported_mask
    threshold_rows.append({
        "threshold": float(threshold),
        "validation_macro_f1": float(f1_score(
            labels[val_idx], end_predictions, labels=ALL_CLASSES,
            average="macro", zero_division=0,
        )),
        "supported_accuracy": float(
            (end_predictions[supported_mask] == labels[val_idx][supported_mask]).mean()
        ),
        "unknown_rejection": float(
            (end_predictions[unknown_mask] == "unknown").mean()
        ),
    })
threshold_frame = pd.DataFrame(threshold_rows)
best_threshold_row = threshold_frame.sort_values(
    ["validation_macro_f1", "supported_accuracy"], ascending=False
).iloc[0]
GATE_THRESHOLD = float(best_threshold_row.threshold)
print("Interim validation threshold:", GATE_THRESHOLD)
display(threshold_frame.sort_values("validation_macro_f1", ascending=False).head(10))


## Identical internal test and automatic V6-C comparison

G1 is measured on the same 1,971 group-separated test windows as V6-C. The
report exports both rows side by side. The locked long-video benchmark remains
untouched during architecture selection.


In [ ]:
gate_test_prob, exercise_test_prob = full_probabilities(test_idx)
gate_test_pred = gate_test_prob.argmax(1)
exercise_test_pred = exercise_test_prob.argmax(1)
end_to_end_pred = np.asarray([
    SUPPORTED_CLASSES[class_id] if probability >= GATE_THRESHOLD else "unknown"
    for probability, class_id in zip(gate_test_prob[:, 1], exercise_test_pred)
])

supported_test_mask = labels[test_idx] != "unknown"
classifier_truth = labels[test_idx][supported_test_mask]
classifier_pred = np.asarray([
    SUPPORTED_CLASSES[index] for index in exercise_test_pred[supported_test_mask]
])
metrics = {
    "experiment": "g1_small_stgcn_comparison",
    "active_supported_class_count": len(SUPPORTED_CLASSES),
    "retired_classes_mapped_to_unknown": sorted(RETIRED_CLASSES),
    "training_archives": [path.name for path in zip_paths],
    "seed": SEED,
    "window_frames": WINDOW,
    "stride_frames": STRIDE,
    "augmentation": AUGMENTATION_CONFIG,
    "node_feature_representation": NODE_FEATURE_CONFIG,
    "input_channels_per_joint": NODE_FEATURES,
    "graph_nodes": 17,
    "skeleton_edges": SKELETON_EDGES,
    "parameter_count": {
        "gate": GATE_PARAMETER_COUNT,
        "exercise": EXERCISE_PARAMETER_COUNT,
        "total": GATE_PARAMETER_COUNT + EXERCISE_PARAMETER_COUNT,
    },
    "sampler": {
        "group_temperature": GROUP_TEMPERATURE,
        "gate": "binary_class_then_sqrt_group_mass_then_window_x_hard_positive_weight",
        "exercise": "canonical_class_then_sqrt_group_mass_then_window",
        "matched_hard_positive_training_windows": len(matched_training_keys),
    },
    "gate_threshold": GATE_THRESHOLD,
    "threshold_selection": "interim group-separated validation; not final calibration",
    "gate": {
        "accuracy": float(accuracy_score(y_gate[test_idx], gate_test_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_gate[test_idx], gate_test_pred)),
        "macro_f1": float(f1_score(
            y_gate[test_idx], gate_test_pred, average="macro", zero_division=0
        )),
    },
    "exercise_classifier_oracle": {
        "accuracy": float(accuracy_score(classifier_truth, classifier_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(classifier_truth, classifier_pred)),
        "macro_f1": float(f1_score(
            classifier_truth, classifier_pred, labels=SUPPORTED_CLASSES,
            average="macro", zero_division=0,
        )),
    },
    "end_to_end": {
        "accuracy": float(accuracy_score(labels[test_idx], end_to_end_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(labels[test_idx], end_to_end_pred)),
        "macro_f1": float(f1_score(
            labels[test_idx], end_to_end_pred, labels=ALL_CLASSES,
            average="macro", zero_division=0,
        )),
    },
    "classes": ALL_CLASSES,
    "supported_classes": SUPPORTED_CLASSES,
}
print(json.dumps(metrics, indent=2))

REPORT_ROOT = Path("/content/calorie_g1_stgcn_comparison_report")
if REPORT_ROOT.exists():
    shutil.rmtree(REPORT_ROOT)
REPORT_ROOT.mkdir()
shutil.copy2(AUGMENTATION_PREVIEW_PATH, REPORT_ROOT / "augmentation_preview.png")

pd.DataFrame(gate_history + exercise_history).to_csv(
    REPORT_ROOT / "training_history.csv", index=False
)
threshold_frame.to_csv(REPORT_ROOT / "gate_threshold_sweep.csv", index=False)
sampler_audit.to_csv(REPORT_ROOT / "sampler_audit.csv", index=False)
(REPORT_ROOT / "metrics.json").write_text(json.dumps(metrics, indent=2))

split_rows = []
for split_name, indices in [("train", train_idx), ("validation", val_idx), ("test", test_idx)]:
    split_rows.extend({
        "window_key": window_keys[index],
        "group_id": groups[index],
        "clip_id": clip_ids[index],
        "source_dataset": sources[index],
        "original_class": original_labels[index],
        "active_class": labels[index],
        "split": split_name,
    } for index in indices)
pd.DataFrame(split_rows).to_csv(REPORT_ROOT / "split_assignments.csv", index=False)

reports = {
    "gate_classification_report.json": classification_report(
        y_gate[test_idx], gate_test_pred, labels=[0, 1],
        target_names=["unknown", "supported"], output_dict=True, zero_division=0,
    ),
    "exercise_classification_report.json": classification_report(
        classifier_truth, classifier_pred, labels=SUPPORTED_CLASSES,
        output_dict=True, zero_division=0,
    ),
    "end_to_end_classification_report.json": classification_report(
        labels[test_idx], end_to_end_pred, labels=ALL_CLASSES,
        output_dict=True, zero_division=0,
    ),
}
for filename, report in reports.items():
    (REPORT_ROOT / filename).write_text(json.dumps(report, indent=2))

plots = [
    ("gate", y_gate[test_idx], gate_test_pred, [0, 1], ["unknown", "supported"]),
    ("exercise", classifier_truth, classifier_pred, SUPPORTED_CLASSES, SUPPORTED_CLASSES),
    ("end_to_end", labels[test_idx], end_to_end_pred, ALL_CLASSES, ALL_CLASSES),
]
for name, truth, prediction, matrix_labels, tick_labels in plots:
    matrix = confusion_matrix(truth, prediction, labels=matrix_labels)
    plt.figure(figsize=(max(6, len(tick_labels) * 0.75), max(5, len(tick_labels) * 0.65)))
    sns.heatmap(
        matrix, annot=True, fmt="d", cmap="Blues",
        xticklabels=tick_labels, yticklabels=tick_labels,
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(name.replace("_", " ").title() + " test confusion matrix")
    plt.tight_layout()
    plt.savefig(REPORT_ROOT / f"{name}_confusion_matrix.png", dpi=180)
    plt.show()

history_frame = pd.DataFrame(gate_history + exercise_history)
for name, group in history_frame.groupby("model"):
    group.plot(
        x="epoch", y=["train_macro_f1", "val_macro_f1"], figsize=(8, 4),
        grid=True, title=name,
    )
    plt.tight_layout()
    plt.savefig(REPORT_ROOT / f"{name}_learning_curve.png", dpi=180)
    plt.show()

pd.DataFrame({
    "window_key": window_keys[test_idx],
    "clip_id": clip_ids[test_idx],
    "source_dataset": sources[test_idx],
    "true_class": labels[test_idx],
    "gate_supported_probability": gate_test_prob[:, 1],
    "exercise_prediction": [SUPPORTED_CLASSES[index] for index in exercise_test_pred],
    "exercise_confidence": exercise_test_prob.max(axis=1),
    "predicted_class": end_to_end_pred,
}).to_csv(REPORT_ROOT / "test_window_predictions.csv", index=False)

torch.save({
    "gate_state_dict": gate_model.state_dict(),
    "exercise_state_dict": exercise_model.state_dict(),
    "classes": ALL_CLASSES,
    "supported_classes": SUPPORTED_CLASSES,
    "retired_classes": sorted(RETIRED_CLASSES),
    "gate_threshold": GATE_THRESHOLD,
    "window_frames": WINDOW,
    "input_channels_per_joint": NODE_FEATURES,
    "node_feature_representation": NODE_FEATURE_CONFIG,
    "graph_nodes": 17,
    "skeleton_edges": SKELETON_EDGES,
    "architecture": "two_stage_small_stgcn_g1",
    "experiment": metrics["experiment"],
}, REPORT_ROOT / "two_stage_stgcn_g1.pt")

comparison_rows = []
for model_name, item in [("v6c_tcn", V6C_METRICS), ("g1_stgcn", metrics)]:
    comparison_rows.append({
        "model": model_name,
        "gate_macro_f1": item["gate"]["macro_f1"],
        "gate_balanced_accuracy": item["gate"]["balanced_accuracy"],
        "oracle_exercise_macro_f1": item["exercise_classifier_oracle"]["macro_f1"],
        "end_to_end_macro_f1": item["end_to_end"]["macro_f1"],
        "end_to_end_balanced_accuracy": item["end_to_end"]["balanced_accuracy"],
        "end_to_end_accuracy": item["end_to_end"]["accuracy"],
    })
comparison_frame = pd.DataFrame(comparison_rows)
comparison_frame.to_csv(REPORT_ROOT / "v6c_tcn_vs_g1_stgcn.csv", index=False)
print("Controlled comparison:")
display(comparison_frame)

zip_path = shutil.make_archive(
    "/content/calorie_g1_stgcn_comparison_report", "zip", root_dir=REPORT_ROOT
)
print("Report ZIP:", zip_path)
print("ZIP size MB:", round(Path(zip_path).stat().st_size / 1024**2, 2))
files.download(zip_path)


## What to return

Return `calorie_g1_stgcn_comparison_report.zip`. We will compare G1 and V6-C on
the identical 1,971 internal test windows, inspect per-class errors, model
complexity, and then apply the frozen provisional calibration protocol. ST-GCN
is promoted only if the improvement is meaningful and survives independent
creator evidence; otherwise V6-C remains the baseline.
